# LoRA 학습 — 충남대 Q&A 시스템 (서버용)

연구실 서버 GPU에서 실행합니다.

- Base: `Qwen/Qwen3-8B` (4bit NF4 양자화)
- LoRA: r=32, alpha=64, dropout=0.1, target=q/k/v/o/up/down_proj
- GPU: Quadro RTX 8000 (48GB, Turing) — **fp16 사용** (bf16 미지원)
- 4bit 양자화 + 큰 배치로 Colab보다 빠르게 학습
- 학습 완료 후 어댑터를 HF Hub에 업로드

## 1. 환경 설정

In [ ]:
import os
import json
import random
import torch

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

# GPU 0번(RTX 8000) 사용
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. HF Hub 로그인 & 데이터 로드

In [ ]:
from huggingface_hub import login

# 실행하면 토큰 입력창이 나옵니다
login()

In [ ]:
HF_REPO = "adoveflash/cnu-qa-system"

# 정제된 학습 데이터 + 청크 매핑 로드
TRAIN_PATH = "data/qa/train_clean.jsonl"
EVAL_PATH = "data/qa/eval.jsonl"
CHUNKS_PATH = "data/corpus/chunks.jsonl"

with open(TRAIN_PATH, encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f if line.strip()]
print(f"학습 샘플 수: {len(train_data)}")

# 검증 데이터
eval_data = []
if os.path.exists(EVAL_PATH):
    with open(EVAL_PATH, encoding="utf-8") as f:
        eval_data = [json.loads(line) for line in f if line.strip()]
    print(f"검증 샘플 수: {len(eval_data)}")

# 청크 매핑 (RAG 컨텍스트 포함 학습용)
chunks_map = {}
if os.path.exists(CHUNKS_PATH):
    with open(CHUNKS_PATH, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                chunk = json.loads(line)
                chunks_map[chunk["chunk_id"]] = chunk["text"]
    print(f"청크 수: {len(chunks_map)}")

## 3. 모델 & 토크나이저 로드

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-8B"

# RTX 8000은 Turing 아키텍처 → bf16 미지원, fp16 사용
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("[1/2] 토크나이저 로드")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("[2/2] 모델 로드 (4bit NF4, fp16)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False  # 학습 시에만 비활성화
model.gradient_checkpointing_enable()

vram_gb = torch.cuda.memory_reserved() / 1024**3
print(f"모델 로드 완료 — VRAM: {vram_gb:.2f} GB")

## 4. LoRA 설정 & 적용

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. 데이터셋 준비

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "당신은 충남대학교 학내 정보 안내 도우미입니다.\n"
    "규칙:\n"
    "1. 반드시 주어진 참고 자료에 있는 정보만 사용하여 답변하세요.\n"
    "2. 참고 자료에 없는 내용은 추측하지 말고 '확인되지 않은 정보입니다'라고 답하세요.\n"
    "3. 답변은 간결하고 정확하게 작성하세요.\n"
    "4. 답변 끝에 출처 URL을 포함하세요."
)
MAX_LENGTH = 768

random.seed(SEED)
random.shuffle(train_data)


def format_chat_messages(qa, chunk_text=""):
    """Q&A 레코드를 chat 메시지 형식으로 변환 (추론과 동일한 형식)."""
    if chunk_text:
        user_content = f"참고 자료:\n{chunk_text}\n\n질문: {qa['question']}"
    else:
        user_content = qa["question"]
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": qa["answer"]},
    ]


def find_assistant_start(tokenizer, messages):
    """assistant 응답 시작 토큰 위치를 찾는다."""
    prompt_only = messages[:2]
    prompt_text = tokenizer.apply_chat_template(
        prompt_only, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    return len(prompt_ids)


def prepare_dataset(data, tokenizer, chunks_map, max_length):
    """컨텍스트 포함 + 답변 토큰에만 loss 적용하는 Dataset 생성."""
    all_input_ids, all_attention_mask, all_labels = [], [], []

    for qa in data:
        chunk_text = chunks_map.get(qa.get("chunk_id", ""), "")
        messages = format_chat_messages(qa, chunk_text)

        full_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
        )
        tokenized = tokenizer(
            full_text,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            add_special_tokens=False,
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]

        # assistant 시작점 이전은 labels=-100 (loss 제외)
        assistant_start = find_assistant_start(tokenizer, messages)
        labels = [-100] * min(assistant_start, len(input_ids))
        labels += input_ids[len(labels):]

        # padding도 -100
        labels = [lb if am == 1 else -100 for lb, am in zip(labels, attention_mask)]
        if len(labels) < max_length:
            labels += [-100] * (max_length - len(labels))
        labels = labels[:max_length]

        all_input_ids.append(input_ids)
        all_attention_mask.append(attention_mask)
        all_labels.append(labels)

    return Dataset.from_dict({
        "input_ids": all_input_ids,
        "attention_mask": all_attention_mask,
        "labels": all_labels,
    })


# 학습 데이터셋
train_dataset = prepare_dataset(train_data, tokenizer, chunks_map, MAX_LENGTH)
print(f"학습 데이터셋: {len(train_dataset)}건")

# 검증 데이터셋
eval_dataset = None
if eval_data:
    eval_dataset = prepare_dataset(eval_data, tokenizer, chunks_map, MAX_LENGTH)
    print(f"검증 데이터셋: {len(eval_dataset)}건")

## 6. 학습

In [ ]:
from transformers import TrainingArguments, Trainer, TrainerCallback
from huggingface_hub import HfApi

OUTPUT_DIR = "models/lora_adapter"
CKPT_DIR = f"{OUTPUT_DIR}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)


class HubUploadCallback(TrainerCallback):
    """에폭마다 HF Hub에 어댑터를 백업한다."""

    def on_save(self, args, state, control, **kwargs):
        print(f"\n에폭 {state.epoch:.0f} → HF Hub 백업 중...")
        try:
            api = HfApi()
            ckpts = sorted(
                [d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")],
                key=lambda x: int(x.split("-")[1]),
            )
            if ckpts:
                latest_path = os.path.join(CKPT_DIR, ckpts[-1])
                api.upload_folder(
                    folder_path=latest_path,
                    path_in_repo="models/lora_adapter",
                    repo_id=HF_REPO,
                )
                print(f"백업 완료: {ckpts[-1]}")
        except Exception as e:
            print(f"백업 실패 (학습은 계속): {e}")


training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16,         # 4bit + 48GB → 배치 크게
    gradient_accumulation_steps=1,          # effective batch = 16
    learning_rate=2e-4,
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="steps" if eval_dataset else "no",
    eval_steps=50 if eval_dataset else None,
    load_best_model_at_end=bool(eval_dataset),
    metric_for_best_model="eval_loss" if eval_dataset else None,
    seed=SEED,
    fp16=True,                              # RTX 8000 (Turing) → fp16
    report_to="none",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[HubUploadCallback()],
)

# 체크포인트에서 이어서 학습
resume_ckpt = None
if os.path.exists(CKPT_DIR):
    ckpts = [d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        resume_ckpt = os.path.join(CKPT_DIR, sorted(ckpts, key=lambda x: int(x.split("-")[1]))[-1])
        print(f"체크포인트에서 재개: {resume_ckpt}")

print("학습 시작")
trainer.train(resume_from_checkpoint=resume_ckpt)
print("학습 완료!")

## 7. 어댑터 저장 & HF Hub 업로드

In [ ]:
# 최종 어댑터 저장
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"어댑터 저장 완료: {OUTPUT_DIR}")

# 저장된 파일 확인
for f in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        print(f"  {f}: {size / 1024**2:.1f} MB")

In [ ]:

api = HfApi()
api.upload_folder(
    folder_path=OUTPUT_DIR,
    path_in_repo="models/lora_adapter",
    repo_id=HF_REPO,
)
print(f"HF Hub 업로드 완료: {HF_REPO}/models/lora_adapter")

## 8. 추론 테스트

In [ ]:
import re

model.eval()

_THINK_TAG_RE = re.compile(r"<think>.*?</think>\s*", flags=re.DOTALL)

def _strip_think_tags(text):
    return _THINK_TAG_RE.sub("", text).strip()

test_questions = [
    "컴퓨터융합학부 졸업 요건이 어떻게 되나요?",
    "수강신청은 언제 하나요?",
    "장학금 신청은 어떻게 하나요?",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    torch.manual_seed(SEED)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.2,
        )
    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    answer = _strip_think_tags(answer)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)